In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

original_model_name = "../../models/PySage"
tokenizer = AutoTokenizer.from_pretrained(original_model_name)
model = AutoModelForCausalLM.from_pretrained(original_model_name)

original_model_dir = original_model_name

# model.save_pretrained(original_model_dir)
# tokenizer.save_pretrained(original_model_dir)


/home/alpha/training-ground/PySage/myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [3]:
import torch
from torch import nn

def count_parameters():
    total = 0
    embedding = 0

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        param_count = param.numel()
        total += param_count

        # Check if it's an embedding parameter (by module type or name)
        if 'embedding' in name.lower() or isinstance(getattr(model, name.split('.')[0], None), nn.Embedding):
            embedding += param_count

    non_embedding = total - embedding

    print(f"Total Parameters: {total:,}")
    print(f"Embedding Parameters: {embedding:,}")
    print(f"Non-Embedding Parameters: {non_embedding:,}")

    return {
        'total': total,
        'embedding': embedding,
        'non_embedding': non_embedding
    }

count_parameters()

Total Parameters: 439,332,864
Embedding Parameters: 0
Non-Embedding Parameters: 439,332,864


{'total': 439332864, 'embedding': 0, 'non_embedding': 439332864}

In [8]:
model.config

Qwen2Config {
  "_attn_implementation_autoset": true,
  "_name_or_path": "../../models/PySage",
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 90869,
  "eos_token_id": 90869,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 90887
}

In [2]:
unique_tokens = set()

class CustomCounter():
    def __init__(self, ):
        self.count = 0

    def increase(self, count):
        self.count += count

total_tokens = CustomCounter()



In [3]:
# updates the unique tokens contained in the text

def tokenizer_function(content):
    if len(content) < 4096:
        for i in range(0, len(content), 4096):
            tokens = tokenizer.encode(content[i: i + 4096])
            total_tokens.increase(len(tokens))

            unique_tokens.update(tokens)



In [4]:
# From python 
import keyword
import builtins

python_tokens = set(keyword.kwlist + dir(builtins))
tokenizer_function(" ".join(python_tokens))

In [5]:
# high quality english dataset
from datasets import load_dataset

ds = load_dataset("agentlans/high-quality-english-sentences")


for split in ['train', 'test']:
    for data in ds[split]:
        tokenizer_function(data['text'])



In [6]:
# from python repository
import os
python_lib_dir = "../../../cpython_lib/Lib"
for root, dirs, files in os.walk(python_lib_dir):
    for file in files:
        file_path = os.path.join(root, file)
        if file.endswith('.py'):  # Adjust file type as needed
            print(file)
            # print(f"Tokenizing file: {file_path}")
            # process file here
            with open(file_path, 'r', errors="ignore") as f:
                tokenizer_function(f.read())

io.py
_android_support.py
_weakrefset.py
tokenize.py
base64.py
_apple_support.py
typing.py
imaplib.py
tabnanny.py
rlcompleter.py
statistics.py
tracemalloc.py
filecmp.py
sre_constants.py
argparse.py
bisect.py
sched.py
warnings.py
zipimport.py
quopri.py
signal.py
string.py
dataclasses.py
_compat_pickle.py
datetime.py
runpy.py
inspect.py
uuid.py
optparse.py
_aix_support.py
numbers.py
csv.py
threading.py
_opcode_metadata.py
ipaddress.py
sre_parse.py
socketserver.py
reprlib.py
turtle.py
py_compile.py
bz2.py
_ios_support.py
fractions.py
types.py
annotationlib.py
getpass.py
decimal.py
_pydecimal.py
_pylong.py
getopt.py
plistlib.py
dis.py
abc.py
mimetypes.py
_compression.py
_osx_support.py
cProfile.py
sre_compile.py
traceback.py
_colorize.py
smtplib.py
pkgutil.py
ssl.py
shlex.py
pprint.py
shutil.py
this.py
tempfile.py
stringprep.py
random.py
configparser.py
_threading_local.py
fnmatch.py
calendar.py
profile.py
codecs.py
operator.py
bdb.py
mailbox.py
contextvars.py
genericpath.py
hmac.py
glob.p

In [7]:
# python repositories scrapped by us
import os
import ujson

scraped_repo_dir = "../../python-dataset/tokenized_scraped_python_code"
files = os.listdir(scraped_repo_dir)
files = [os.path.join(scraped_repo_dir, file) for file in files]

for file in files:
    with open(file, 'r') as f:
        for l in f:
            data = ujson.loads(l)
            unique_tokens.update(data["tokens"])
            total_tokens.increase(len(data["tokens"]))



In [8]:
print(total_tokens.count)

1024840245


In [9]:
print(len(unique_tokens))

90887


In [ ]:
# Model completion
import ujson

model_response = ''
with open(model_response, 'r') as f:
    for l in f:
        data = ujson.loads(l)
        tokenizer_function(data['completion'])

In [ ]:
# custom ones
to_remove = {
  "151648": "<|box_start|>",
  "151649": "<|box_end|>",
  "151652": "<|vision_start|>",
  "151653": "<|vision_end|>",
  "151654": "<|vision_pad|>",
  "151655": "<|image_pad|>",
  "151656": "<|video_pad|>",
}
for k, v in to_remove.items():
    unique_tokens.add(int(k))
    


## Save the Unique tokens collected

In [11]:
import json
# ids to token mapping
with open("unique_tokens.json", "w", encoding="utf-8") as f:
    unique_token_dict = {}
    for token in unique_tokens:
        unique_token_dict[token] = tokenizer.convert_ids_to_tokens(token)
    json.dump(unique_token_dict, f, ensure_ascii=False)

In [ ]:
from transformers import AutoTokenizer
import json

with open("changed_vocab_mapping.json", 'r') as f:
    changed_vocab_mapping = json.loads(f.read())
    with open("unique_tokens.json", "w") as f:
        f.write(json.dumps({k: tokenizer.convert_ids_to_tokens(int(k)) for k, v in changed_vocab_mapping.items()}))

In [4]:
import ujson
unique_tokens_dict = {}
with open("unique_tokens.json", 'r') as f:
    unique_tokens_dict = ujson.loads(f.read())

# changed_vocab_mapping = {old: new for new, old in enumerate(unique_token_dict.keys())}
# # also save updated id
# with open("changed_vocab_mapping.json", "w") as f:
#     f.write(ujson.dumps(changed_vocab_mapping))

In [38]:
unique_tokens_dict

{'0': '!',
 '1': '"',
 '2': '#',
 '3': '$',
 '4': '%',
 '5': '&',
 '6': "'",
 '7': '(',
 '8': ')',
 '9': '*',
 '10': '+',
 '11': ',',
 '12': '-',
 '13': '.',
 '14': '/',
 '15': '0',
 '16': '1',
 '17': '2',
 '18': '3',
 '19': '4',
 '20': '5',
 '21': '6',
 '22': '7',
 '23': '8',
 '24': '9',
 '25': ':',
 '26': ';',
 '27': '<',
 '28': '=',
 '29': '>',
 '30': '?',
 '31': '@',
 '32': 'A',
 '33': 'B',
 '34': 'C',
 '35': 'D',
 '36': 'E',
 '37': 'F',
 '38': 'G',
 '39': 'H',
 '40': 'I',
 '41': 'J',
 '42': 'K',
 '43': 'L',
 '44': 'M',
 '45': 'N',
 '46': 'O',
 '47': 'P',
 '48': 'Q',
 '49': 'R',
 '50': 'S',
 '51': 'T',
 '52': 'U',
 '53': 'V',
 '54': 'W',
 '55': 'X',
 '56': 'Y',
 '57': 'Z',
 '58': '[',
 '59': '\\',
 '60': ']',
 '61': '^',
 '62': '_',
 '63': '`',
 '64': 'a',
 '65': 'b',
 '66': 'c',
 '67': 'd',
 '68': 'e',
 '69': 'f',
 '70': 'g',
 '71': 'h',
 '72': 'i',
 '73': 'j',
 '74': 'k',
 '75': 'l',
 '76': 'm',
 '77': 'n',
 '78': 'o',
 '79': 'p',
 '80': 'q',
 '81': 'r',
 '82': 's',
 '83': 't',
 

## Modify and Save the model

In [16]:
import torch
import os
import shutil
import json
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer



def prune_model(model_dir, used_token_indices):
    """
    Prune the model and tokenizer to keep only specified tokens.
    
    Args:
        model: Huggingface model dir 
        used_token_indices: List of token indices to keep
    
    Returns:
        Tuple of (pruned model, pruned tokenizer)
    """
    # Ensure indices are sorted and unique
    used_token_indices = sorted(map(int, used_token_indices))
    # used_token_indices
    # print(used_token_indices)
    # Load the model
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    
    
    # Prune the embedding layer
    embedding_layer = model.get_input_embeddings()
    embedding_matrix = embedding_layer.weight.data
    new_embedding_matrix = embedding_matrix[used_token_indices]
    embedding_layer.weight.data = new_embedding_matrix
    
    # Update the model config
    model.config.vocab_size = len(used_token_indices)
    for i, token in enumerate(used_token_indices):
        if token == model.config.eos_token_id:
            model.config.eos_token_id = i
        if token == model.config.bos_token_id:
            model.config.bos_token_id = i
            model.generation_config.bos_token_id = i
        if token == model.generation_config.pad_token_id:
            model.generation_config.pad_token_id  = i
        if token ==  model.generation_config.eos_token_id:
            model.generation_config.eos_token_id = i
            
    
    return model

def save_pruned_model(model, save_dir):
    """
    Save the pruned model and tokenizer
    
    Args:
        model: Pruned model
        tokenizer: Pruned tokenizer
        save_dir: Directory to save the model and tokenizer
    """
    
    os.makedirs(save_dir, exist_ok=True)
    
    # Save the model and tokenizer
    model.save_pretrained(save_dir)
    
    print(f"Model saved to {save_dir}")
    print(f"Updated vocab size: {model.config.vocab_size}")

### Modify the tokenizers Confguration

In [12]:
# Re-write merges.txt and vocab.json
import os
import json


def prune_tokenizer(tokenizer_dir, unique_tokens_indices, save_dir) -> AutoTokenizer:
    """
    Prune the model by modifying the merges.txt, vocab.json, tokenizer.json, tokenizer_config.json

    Args: 
        tokenizer_dir: Path to the original tokenizer
        unique_tokens_indices: Set of unique tokens to be kept
        save_dir: Directory to save the pruned model
    """
    # Old Paths
    merges_path = os.path.join(tokenizer_dir, "merges.txt")
    tokenizer_config_path = os.path.join(tokenizer_dir, "tokenizer_config.json")
    tokenizer_json_path = os.path.join(tokenizer_dir, "tokenizer.json")
    vocab_path = os.path.join(tokenizer_dir, "vocab.json")
    special_tokens_path = os.path.join(tokenizer_dir, "special_tokens_map.json")
    added_tokens_path =  os.path.join(tokenizer_dir, "added_tokens.json")

    original_tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
    original_vocab = original_tokenizer.get_vocab()

    # Make the directory if it already doesn't exists
    os.makedirs(save_dir, exist_ok=True)

    # New Paths
    new_merges_path = os.path.join(save_dir, 'merges.txt')
    new_tokenizer_config_path = os.path.join(save_dir, "tokenizer_config.json")
    new_toknizer_json_path = os.path.join(save_dir, "tokenizer.json")
    new_vocab_path = os.path.join(save_dir, "vocab.json")
    new_special_tokens_path = os.path.join(save_dir, "special_tokens_map.json")
    new_added_tokens_path = os.path.join(save_dir, "added_tokens.json")
    # Mapping of old token id to new
    old_to_new = {old: i for i, old in enumerate(sorted(unique_tokens_indices, key=int))}
    old_id_to_tok = {str(v): k for k, v in original_vocab.items()}

    # Modify merges.txt
    with open(merges_path, 'r') as f: 
        mrules = f.read().split('\n')[1:-1]

    new_mrules = []
    for _merge in mrules:
        m1, m2 = _merge.split(" ")

        m1_m2_id = str(original_vocab[m1 + m2])
        m1_id = str(original_vocab[m1])
        m2_id = str(original_vocab[m2])
        if m1_m2_id in unique_tokens_indices and m1_id in unique_tokens_indices and m2_id in unique_tokens_indices:
            new_mrules.append(_merge)
   

    with open(new_merges_path, 'w') as f:
        f.write("#version: 0.2\n")
        f.writelines([mrule + '\n' for mrule in new_mrules])
    
    # Modify vocab.json
    with open(vocab_path, 'r') as f:
        new_vocab = json.load(f)
    
    new_vocab = {k: old_to_new[str(v)] for k, v in new_vocab.items() if str(v) in old_to_new}
    with open(new_vocab_path, 'w') as f:
        json.dump(new_vocab, fp = f, ensure_ascii=False)

    
    # Modify tokenizer.json
    with open(tokenizer_json_path , "r") as f: tokenizer_json = json.load(f)
    tokenizer_json["model"]["vocab"] = {k: old_to_new[str(v)] for k, v in tokenizer_json["model"]["vocab"].items() if str(v) in old_to_new} 

    tokenizer_json['model']["merges"] = [mrule.split(" ") for mrule in new_mrules]

    new_added_tokens = []
    for added_token in tokenizer_json["added_tokens"]:
        if str(added_token["id"]) not in unique_tokens_indices:
            continue
        
        added_token["id"] = old_to_new[str(added_token["id"])]
        new_added_tokens.append(added_token)
    
    tokenizer_json["added_tokens"] = new_added_tokens

    with open(new_toknizer_json_path, "w", encoding='utf-8') as f:
        json.dump(tokenizer_json, f, ensure_ascii=False)

    # Modify tokenizer_config.json
    with open(tokenizer_config_path, 'r') as f:
        tokenizer_config = json.load(f)

    new_added_tokens_decoder = {}
    for added_token_id in tokenizer_config["added_tokens_decoder"].keys():
        if added_token_id not in old_to_new:
            continue
        
        new_added_tokens_decoder[old_to_new[added_token_id]] = tokenizer_config["added_tokens_decoder"][added_token_id]
    tokenizer_config['added_tokens_decoder'] = new_added_tokens_decoder

    # modify the additional_special_tokens 
    new_additional_special_tokens = []
    for additional_special_token in tokenizer_config["additional_special_tokens"]:
        if str(original_vocab[additional_special_token])not in unique_tokens_indices:
            continue
        new_additional_special_tokens.append(additional_special_token)


        
    tokenizer_config['additional_special_tokens'] = new_additional_special_tokens

    with open(new_tokenizer_config_path, 'w', encoding='utf-8') as f:
        json.dump(tokenizer_config, f,  ensure_ascii=False)
    
    # Modify special_tokens_map.json
    with open(special_tokens_path, "r") as f:
        special_tokens_map = json.load(f)
    
    new_additional_special_tokens = []

    for tok in special_tokens_map["additional_special_tokens"]:
        if str(original_vocab[tok]) not in unique_tokens_indices:
            continue

        new_additional_special_tokens.append(tok)

    special_tokens_map["additional_special_tokens"] = new_additional_special_tokens

    with open(new_special_tokens_path, 'w') as f:
        json.dump(special_tokens_map, f, ensure_ascii=False)
    
    # Modify added_tokens.json

    with open(added_tokens_path, 'r') as f:
        added_tokens_map = json.load(f)

    new_added_tokens_map = {}
    for tok, idx in added_tokens_map.items():
        if str(idx) not in unique_tokens_indices:
            continue
        new_added_tokens_map[tok] = old_to_new[str(idx)]
    
    with open(new_added_tokens_path, 'w') as f:
        json.dump(new_added_tokens_map, f, ensure_ascii=False)



       

In [17]:
original_model_dir="../../models/Qwen2.5-Coder-0.5B"
custom_model_save_dir="../../models/PySage"
unique_tokens_indices = set(unique_tokens_dict.keys())

# # Delete the directory if already present
# if os.path.exists(custom_model_save_dir):
#     print("The directory exists")
#     print("Deleting the existing directory and recreating the directory")
#     shutil.rmtree(custom_model_save_dir) 
#     os.makedirs(custom_model_save_dir, exist_ok=True)
    

model = prune_model(original_model_dir, unique_tokens_indices)
save_pruned_model(model, custom_model_save_dir)

# prune_tokenizer(original_model_dir, unique_tokens_indices, custom_model_save_dir)

Model saved to ../../models/PySage
Updated vocab size: 90887


### Load the original tokenizer an dmodel

In [33]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained(original_model_dir)

In [77]:
# load the tokenizer
from transformers import AutoTokenizer


custom_tokenizer = AutoTokenizer.from_pretrained(custom_model_save_dir)

In [78]:
custom_tokenizer("<|fim_prefix|>")

{'input_ids': [90881], 'attention_mask': [1]}

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

original_model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

original_model = AutoModelForCausalLM.from_pretrained(original_model_name)
original_tokenizer = AutoTokenizer.from_pretrained(original_model_name)


In [20]:
original_vocab = tokenizer.get_vocab()

In [ ]:
# Re-write merges.txt and vocab.json
import os
tokenizer_path = "../../models/Qwen2.5-Coder-0.5B-Instruct-GGUF"
with open(os.path.join(tokenizer_path, "merges.txt"), 'r') as f: mrules = f.read().split('\n')[1:-1]

import json


for i, _merge in enumerate(mrules):
    merge = _merge.replace(' ', '')
    idx = original_vocab[merge]
    if idx not in unique_token_dict: mrules.remove(_merge)

with open('merges.txt', 'w') as f:
    f.write("#version: 0.2\n")
    
    f.writelines([mrule + '\n' for mrule in mrules])


In [ ]:
with open(os.path.join(tokenizer_path, 'vocab.json'), 'r') as f: vocab = json.loads(f.read())
new_vocab ={original_tokenizer.convert_ids_to_tokens(int(token_id)): idx for idx, token_id in enumerate(sorted(unique_token_dict.keys(), key=int))} 
with open("new_vocab.json", "w", encoding="utf-8") as f: json.dump(new_vocab, fp=f, ensure_ascii=True)

In [ ]:
import os
import json
tokenizer_path = "../../models/Qwen2.5-Coder-0.5B-Instruct-GGUF" # original tokenizer path


with open(os.path.join(tokenizer_path, "tokenizer.json" ), "r") as f: tokenizer_config = json.load(f)
with open('./new_vocab.json', 'r') as f:
    new_vocab = json.load(fp=f)

# Modify the vocabulary 
tokenizer_config["model"]["vocab"] = new_vocab 

with open("./merges.txt", "r") as f: mrules = f.read().split("\n")[1:-1]

mrules_list = [ [merge.split()[0], merge.split()[1] ] for merge in mrules]
tokenizer_config['model']["merges"] = mrules_list
with open("tokenizer_config.json", "w", encoding='utf-8') as f:
    json.dump(tokenizer_config, f, ensure_ascii=False)

In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

def benchmark_model(model, tokenizer, device, prompt, max_new_tokens=100, num_runs=5):
    model = model.to(device)
    model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Warmup run
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=max_new_tokens)

    total_time = 0

    for _ in range(num_runs):
        start_time = time.time()
        
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens)
        
        end_time = time.time()
        
        total_time += end_time - start_time

    avg_time = total_time / num_runs
    num_generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    
    throughput = num_generated_tokens / avg_time
    latency = avg_time / num_generated_tokens

    return {
        "avg_time_per_generation": avg_time,
        "tokens_generated": num_generated_tokens,
        "throughput_tokens_per_sec": throughput,
        "latency_per_token": latency,
    }


def run_benchmark(old_model, old_tokenizer, pruned_model, pruned_tokenizer, device, prompt, max_new_tokens=100, num_runs=5):
    print("Benchmarking old model...")
    old_metrics = benchmark_model(old_model, old_tokenizer, device, prompt, max_new_tokens, num_runs)
    
    torch.cuda.empty_cache()  # Free up GPU memory
    
    print("\nBenchmarking pruned model...")
    pruned_model = pruned_model.to("cuda") if torch.cuda.is_available() else pruned_model
    pruned_metrics = benchmark_model(pruned_model, pruned_tokenizer, device, prompt, max_new_tokens, num_runs)
    
    print("\nBenchmark Results:")
    print(f"{'Metric':<30}{'Old Model':<20}{'Pruned Model':<20}")
    print("=" * 70)
    for key in old_metrics:
        old_value = old_metrics[key]
        pruned_value = pruned_metrics[key]
        print(f"{key:<30}{old_value:<20.4f}{pruned_value:<20.4f}")


# Example usage
prompt = "Once upon a time, in a land far away, there was a wise old sage who"
pruned_model_dir = "../../models/PySage"
original_model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

original_model = AutoModelForCausalLM.from_pretrained(original_model_name)
original_tokenizer = AutoTokenizer.from_pretrained(original_model_name)

our_model = AutoModelForCausalLM.from_pretrained(pruned_model_dir)
our_tokenizer = AutoTokenizer.from_pretrained(pruned_model_dir)

run_benchmark(
    original_model, original_tokenizer,
    our_model, our_tokenizer,
    torch.device('cpu'),
    prompt,
    max_new_tokens=100,
    num_runs=2,
)


Benchmarking old model...

Benchmarking pruned model...

Benchmark Results:
Metric                        Old Model           Pruned Model        
avg_time_per_generation       13.7956             12.3509             
tokens_generated              100.0000            100.0000            
throughput_tokens_per_sec     7.2487              8.0966              
latency_per_token             0.1380              0.1235              


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
custom_model_dir = "../../models/PySage"

our_model = AutoModelForCausalLM.from_pretrained(custom_model_dir)
our_tokenizer = AutoTokenizer.from_pretrained(custom_model_dir)

In [15]:
tokens = our_tokenizer("Write a function to generate fibonacci series.\n", return_tensors="pt")

output = our_model.generate(**tokens, max_new_tokens = 512)

print(our_tokenizer.decode(output[0], skip_special_tokens=True))

Write a function to generate fibonacci series.
```python
def fibonacci_series(n):
    """
    Generate Fibonacci series up to the nth number.

    Args:
    n (int): The number of terms in the Fibonacci series to generate.

    Returns:
    list: A list containing the Fibonacci series up to the nth term.
    """
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    else:
        series = [0, 1]
        for i in range(2, n):
            next_term = series[i-1] + series[i-2]
            series.append(next_term)
        return series

# Test the function
print(fibonacci_series(10))  # Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
```
This solution provides a concise and efficient way to generate Fibonacci series in Python. It includes a function that takes an integer `n` as input and returns a list of Fibonacci numbers up to that number. The function handles edge cases such as when `n` is less than or equal to 0, returning an empty list. For `n` greater than 1, it initializ

In [16]:
def fibonacci_series(n):
    """
    Generate Fibonacci series up to the nth number.

    Args:
    n (int): The number of terms in the Fibonacci series to generate.

    Returns:
    list: A list containing the Fibonacci series up to the nth term.
    """
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    else:
        series = [0, 1]
        for i in range(2, n):
            next_term = series[i-1] + series[i-2]
            series.append(next_term)
        return series

# Test the function
print(fibonacci_series(10))  # Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
